# Storage and Executors
In this example we're covering a finer grained control over the workflow run.
These are the internals that are set inside the `run_workflow` function.
For this we're going to construct a graph thad runs a `pytket` circuit on a simulator using the `quantinuum_worker`
The worker can be installed using `pip install tkr-quantinuum-worker`

## Opaque Types
Tierkreis can use any type that is serializable as in and outputs.
To use such types for type hinting you can use the `OpaqueType`.

In [1]:
%pip install tierkreis pytket

/Users/philipp.seitz/Projects/tierkreis/.devenv/state/venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from tierkreis.controller.data.models import OpaqueType
Circuit = OpaqueType["pytket._tket.circuit.Circuit"]
BackendResult = OpaqueType["pytket.backends.backendresult.BackendResult"]

Now we can construct a graph using the `quantinuum_worker`.
The api definitions for workers build by the tierkreis teams are already included in tierkreis.
Still it is necessary to install the worker manually.
We define a graph `Circuit -> BackendResult` using hardcoded information for which emulator backend to use.

In [3]:
from tierkreis.builder import GraphBuilder
from tierkreis.controller.data.models import TKR
from tierkreis.quantinuum_worker import (
    get_backend_info,
    compile_using_info,
    run_circuit,
)
g = GraphBuilder(TKR[Circuit], TKR[BackendResult])
info = g.task(get_backend_info(device_name=g.const("H2-1")))
compiled_circuit = g.task(compile_using_info(g.inputs, info))
results = g.task(run_circuit(circuit=compiled_circuit, n_shots=g.const(10), device_name=g.const("H2-1SC")))
g.outputs(results)

Now we will define our own storage and executor.
Storage is responsible for setting up the checkpointing; it stores the state of the computation.
We have to provide a uuid, and optionally a name, as before.

In [4]:
from uuid import UUID
from tierkreis.storage import FileStorage
storage = FileStorage(UUID(int=209), do_cleanup=True, name="quantinuum_submission")

Since the `quantinuum_worker` is a python worker we will use `uv` to run it.
An executor lives in context, where it can access workers to run their `main` entrypoints.
We define this by providing a path to the directory our workers live in, in this case in the `tierkreis_workers` directory.


In [5]:
from pathlib import Path
from tierkreis.consts import PACKAGE_PATH
from tierkreis.executor import UvExecutor
executor = UvExecutor(
    PACKAGE_PATH.parent / "tierkreis_workers" , storage.logs_path
)

Since this graph is using the `qnexus` api internally you also need to run the following once: 

In [ ]:
from qnexus.client.auth import login
login()

SyntaxError: invalid syntax (3725152821.py, line 1)

Once we provide the graph inputs we can now run it by providing a storage and an executor.

In [6]:
from tierkreis import run_graph
from pytket.qasm.qasm import circuit_from_qasm
circuit = circuit_from_qasm(Path().parent / "data" / "ghz_state_n23.qasm")
run_graph(storage, executor, g, circuit)

Placing the launcher in the root directory is deprecated.
 Please move it to a 'src' subdirectory.


.py


Placing the launcher in the root directory is deprecated.
 Please move it to a 'src' subdirectory.


.py


Placing the launcher in the root directory is deprecated.
 Please move it to a 'src' subdirectory.


.py


And finally we can print the outputs

In [7]:
from tierkreis.storage import read_outputs
outputs = read_outputs(g, storage)
print(outputs)


{'qubits': [], 'bits': [['meas', [22]], ['meas', [21]], ['meas', [20]], ['meas', [19]], ['meas', [18]], ['meas', [17]], ['meas', [16]], ['meas', [15]], ['meas', [14]], ['meas', [13]], ['meas', [12]], ['meas', [11]], ['meas', [10]], ['meas', [9]], ['meas', [8]], ['meas', [7]], ['meas', [6]], ['meas', [5]], ['meas', [4]], ['meas', [3]], ['meas', [2]], ['meas', [1]], ['meas', [0]], ['c', [22]], ['c', [21]], ['c', [20]], ['c', [19]], ['c', [18]], ['c', [17]], ['c', [16]], ['c', [15]], ['c', [14]], ['c', [13]], ['c', [12]], ['c', [11]], ['c', [10]], ['c', [9]], ['c', [8]], ['c', [7]], ['c', [6]], ['c', [5]], ['c', [4]], ['c', [3]], ['c', [2]], ['c', [1]], ['c', [0]]], 'shots': {'width': 46, 'array': [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]]}}
